# ep0 大模型推理（NVIDIA GPU / CUDA 版）

本 Notebook 为 `exp2.ipynb`（原昇腾 NPU 版）的 **NVIDIA GPU 适配版**，保留原文件不做修改，底层调用同目录下的 `cuda_inference.py`。

核心改动：
- 移除 `torch_npu` 与 ACL/NPU 依赖，使用 PyTorch CUDA + `sdpa` 注意力 + `bfloat16`/`float16` 精度
- 在 `torch.inference_mode()` 下进行增量 KV-Cache 推理，避免无意义梯度显存开销
- 保持与原实验一致的采样策略对比（Greedy / Top-K / Top-P）和不同输入输出长度测试，并统计 TTFT、Decode 吞吐量与峰值显存

In [ ]:
import torch
from cuda_inference import CudaInferenceConfig, CudaInference

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. 初始化 CUDA 推理配置与模型会话

In [ ]:
config = CudaInferenceConfig(
    hf_model_dir="../data/Qwen2.5-3B-Instruct",
    device_str="cuda:0",
    torch_dtype="bfloat16",
    max_input_length=1024,
    max_output_length=2048,
)

inference = CudaInference(config)

## 2. 基础推理测试

In [ ]:
prompt = "What is the capital of France?"
output, stats = inference.predict(prompt, return_stats=True)
print("Generated Output:", output)
print("Stats:", stats)

## 3. 采样策略对比（Greedy / Top-K / Top-P）

In [ ]:
prompt = "What is the capital of France?"
output_greedy = inference.predict(prompt, sampling_method="greedy")
print("Greedy Sampling Output:")
print(output_greedy)

In [ ]:
prompt = "What are the benefits of artificial intelligence?"
output_top_k = inference.predict(prompt, sampling_method="top_k", sampling_value=5, temperature=0.8, max_new_tokens=256)
print("Top-K Sampling Output:")
print(output_top_k)

In [ ]:
prompt = "Can you explain quantum computing?"
output_top_p = inference.predict(prompt, sampling_method="top_p", sampling_value=0.9, temperature=0.8, max_new_tokens=256)
print("Top-P Sampling Output:")
print(output_top_p)

## 4. 不同输入/输出长度测试

In [ ]:
prompt = "What is 2 + 2?"
output_short = inference.predict(prompt, max_new_tokens=64)
print("Short Input/Output Example:")
print(output_short)

In [ ]:
prompt = "Can you explain how quantum entanglement works and its potential applications in future technologies?"
output_long = inference.predict(prompt, max_new_tokens=64)
print("Long Input/Output Example:")
print(output_long)

In [ ]:
prompt = "Can you give me a detailed explanation of machine learning algorithms, including supervised learning, unsupervised learning, and reinforcement learning, and provide examples of each?"
output_extreme = inference.predict(prompt, max_new_tokens=128)
print("Extreme Length Input/Output Example:")
print(output_extreme)

## 5. 释放 GPU 显存资源

In [ ]:
inference.close()